# Airbnb Price Model Performance Under Distribution Shift

This notebook represents an updated version of an earlier analysis, with a focus on model robustness and performance under distribution shift.

This data taken from the official website Airbnb http://insideairbnb.com/get-the-data.html

## Business Understanding

This project studies Airbnb price prediction under realistic data constraints. Only the most recent twelve months of data are available, which reflects a common production scenario where historical archives are limited or costly to access.

The goal is to evaluate how machine learning models behave when trained and validated within a narrow time window. I focus on model stability, feature reliability, and performance tradeoffs when long-term historical context is unavailable.

The analysis addresses the following questions:

1. How stable are price prediction models when trained on a limited twelve-month window of recent Airbnb data?

2. Which listing features remain reliable predictors of price within short time horizons, and which degrade as market conditions shift?

3. How do more complex models compare to simple baselines when data is constrained in both time and volume?

4. At what point does increasing model complexity improve price prediction, and when does it introduce noise or overfitting instead?

These questions reflect practical constraints faced by production machine learning systems, where historical data access is limited, market behavior changes quickly, and modeling decisions must balance performance, stability, and maintainability.

Let's begin!

In [1]:
#import libraries
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
def printColunmsInfo(df):
    '''takes dataframe, prints columns info'''
    df.info()
    print("\n")
    printTotalRowsAndColumns(df)
    print("---------------------------------------")
    
def printTotalRowsAndColumns(df):
    '''print number of columns and rows'''
    print("Total columns: ", df.shape[1])
    print("Total rows: ", df.shape[0])

In [5]:
def stringToNumConverter(string):
    '''deletes not numbers symbols from string'''
    newString = ""
    if pd.notna(string):
        for i in string:
            if i.isdigit() or i == ".":
                newString += i            
    return newString

In [6]:
def create_dummy_df(df, cat_cols, dummy_na):
    '''creates dummy'''
    for col in  cat_cols:
        try:
            # for each cat add dummy var, drop original column
            df = pd.concat([df.drop(col, axis=1), pd.get_dummies(df[col], prefix=col, prefix_sep='_', drop_first=True, dummy_na=dummy_na)], axis=1)
        except:
            continue
    return df

In [7]:
def dateToCategorical(row):
    '''changes column from date type to categorical'''
    if row.year <= 2016:
        return "4+ years"
    elif row.year <= 2018:
        return "2-3 years"
    elif row.year <= 2019:
        return "1-2 years"
    elif row.year == 2020:
        if row.month > 8:
            return "0-1 month"
        elif row.month > 2:
            return "2-6 months"
        elif row.month <= 2:
            return "this year"
    else :
        return "no reviews"   

In [8]:
#load data
sf_cal = pd.read_csv("datasets/calendar.csv", low_memory=False, index_col=0)
sf_list = pd.read_csv("datasets/listings.csv")

## Cleaning the Data

Data cleaning logic is reused from the previous version of the project to ensure consistency across experiments.

### Listing Data Frame

First, let's look on Listing Data Frame. It is the biggest table. We won't need some columns because they don't make much sense for our purposes. So we will drop them.

In [9]:
sf_list = sf_list[['id', 'host_since', 'host_is_superhost', 'host_listings_count', 'host_response_time', 
                   'host_response_rate', 'host_acceptance_rate','neighbourhood_cleansed', 'latitude', 'longitude', 
                   'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'amenities', 
                   'minimum_nights', 'maximum_nights', 'review_scores_rating', 'review_scores_accuracy', 
                   'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 
                   'review_scores_location', 'review_scores_value', 'availability_30', 'number_of_reviews',
                   'last_review', 'reviews_per_month', 'price']]
sf_list.head()

,id,host_since,host_is_superhost,host_listings_count,host_response_time,host_response_rate,host_acceptance_rate,neighbourhood_cleansed,latitude,longitude,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,availability_30,number_of_reviews,last_review,reviews_per_month,price
0,958,2008-07-31,t,1,within an hour,100%,99%,Western Addition,37.76931,-122.43386,...,10.0,10.0,10.0,10.0,10.0,10,257,2020-07-30,1.91,$132.00
1,5858,2009-03-02,f,2,within a day,60%,83%,Bernal Heights,37.74511,-122.42102,...,10.0,10.0,10.0,10.0,9.0,30,111,2017-08-06,0.81,$235.00
2,7918,2009-06-17,f,10,within a few hours,100%,100%,Haight Ashbury,37.76555,-122.45213,...,8.0,9.0,9.0,9.0,8.0,30,19,2020-03-06,0.14,$56.00
3,8142,2009-06-17,f,10,within a few hours,100%,100%,Haight Ashbury,37.76555,-122.45213,...,9.0,10.0,10.0,9.0,9.0,30,8,2018-09-12,0.11,$56.00
4,8339,2009-07-02,f,2,within a few hours,100%,0%,Western Addition,37.77525,-122.43637,...,10.0,10.0,10.0,10.0,10.0,30,28,2019-06-28,0.21,$750.00


We have left the following columns:

* __'id'__ — we'll use to join tables
* __host_since__ and __last_review__ — datatype data, we transform to categorical
* __'host_response_time'__ — categorical data
* __host_is_superhost__ — boolean data
* __'host_response_rate'__ and __'host_acceptance_rate'__ — as a percentage, we will change to integer
* __neighbourhood_cleansed'__ — neighbourhood name
* __'latitude', 'longitude'__ — сoordinates, we use them for visualisation
* __'room_type'__ and __property_type__ — categorical data
* __'accommodates', 'bathrooms', 'bedrooms', 'beds'__ — numerical values describing property
* __'amenities'__ — can be used to identify words associated with amenities
* __'minimum_nights', 'maximum_nights'__ — numerical values
* __'review_scores_rating'__ — numbers between 20 and 100
* __'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', review_scores_communication', 'review_scores_location', 'review_scores_value'__ — numbers between 2 and 10
* __availability_30__, __number_of_reviews__, __reviews_per_month__ — numerical
* __'price'__ — target value

Let's convert string data to numeric.

In [10]:
#converting datatype of price column to integer
sf_list["price"] = sf_list["price"].apply(lambda string: ''.join(i for i in string if i.isdigit())[:-2])
sf_list["price"] = pd.to_numeric(sf_list["price"], downcast="integer")

#host_response_rate and host_acceptance_rate types to float
sf_list["host_response_rate"] = sf_list["host_acceptance_rate"].apply(lambda string: stringToNumConverter(string))
sf_list["host_response_rate"] = pd.to_numeric(sf_list["host_response_rate"], downcast="float")

sf_list["host_acceptance_rate"] = sf_list["host_acceptance_rate"].apply(lambda string: stringToNumConverter(string))
sf_list["host_acceptance_rate"] = pd.to_numeric(sf_list["host_acceptance_rate"], downcast="float")

#converting t, f value to 1 or 0
sf_list["host_is_superhost"] = sf_list["host_is_superhost"].apply((lambda string: 1 if string == "t" else 0))

#converting datatype of date columns to datetime
sf_list["last_review"] = pd.to_datetime(arg=sf_list["last_review"], errors="coerce")
sf_list["host_since"] = pd.to_datetime(arg=sf_list["host_since"], errors="coerce")


In [11]:
print("Listing Data Frame")
printColunmsInfo(sf_list)

Listing Data Frame
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7274 entries, 0 to 7273
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   id                           7274 non-null   int64         
 1   host_since                   7274 non-null   datetime64[ns]
 2   host_is_superhost            7274 non-null   int64         
 3   host_listings_count          7274 non-null   int64         
 4   host_response_time           5321 non-null   object        
 5   host_response_rate           6393 non-null   float32       
 6   host_acceptance_rate         6393 non-null   float32       
 7   neighbourhood_cleansed       7274 non-null   object        
 8   latitude                     7274 non-null   float64       
 9   longitude                    7274 non-null   float64       
 10  property_type                7274 non-null   object        
 11  room_type               

### Amenities Data Frame

In [12]:
amenitiesList = []
for item in sf_list["amenities"].value_counts().reset_index().iloc[:, 0]:
    item = eval(item)
    for i in item:
        if i not in amenitiesList:
            amenitiesList.append(i)
print("Total amenities: ", len(amenitiesList))
print(amenitiesList)

Total amenities:  131
['Gym', 'Heating', 'Hot water', 'Air conditioning', 'Iron', 'Dryer', 'Carbon monoxide alarm', 'Building staff', 'Luggage dropoff allowed', 'Private entrance', 'Microwave', 'Hangers', 'Essentials', 'Smoke alarm', 'Refrigerator', 'Wifi', 'Shampoo', 'TV', 'Hair dryer', 'Washer', 'Cable TV', 'Bed linens', 'Stove', 'Coffee maker', 'Elevator', 'Laptop-friendly workspace', 'Extra pillows and blankets', 'Cooking basics', 'Dishes and silverware', 'Long term stays allowed', 'Kitchen', 'Oven', 'Dishwasher', 'BBQ grill', 'Host greets you', 'Free street parking', 'Hot tub', 'First aid kit', 'Fire extinguisher', 'Lock on bedroom door', 'Paid parking on premises', 'Lockbox', 'Bathtub', 'Garden or backyard', 'Patio or balcony', 'Indoor fireplace', 'Keypad', 'Cleaning before checkout', 'Paid parking off premises', 'Smart lock', 'Free parking on premises', 'Breakfast', 'Bread maker', 'Shower gel', 'Crib', 'Baking sheet', 'Pool', 'Pack ’n Play/travel crib', 'Ethernet connection', 'P

In [13]:
amenities_df = sf_list[["id", "amenities"]].copy()

In [14]:
from __future__ import annotations
import json

def parse_amenities_json(x) -> set[str]:
    if pd.isna(x):
        return set()
    if isinstance(x, list):
        return set(x)

    s = str(x).strip()
    if not s:
        return set()

    try:
        val = json.loads(s)
    except json.JSONDecodeError:
        return {s}

    return set(val) if isinstance(val, list) else {str(val)}


def add_amenity_indicator_columns(
    amenities_df: pd.DataFrame,
    amenities_list: list[str],
    amenities_col: str = "amenities",
) -> pd.DataFrame:
    df = amenities_df.copy()

    amenities_sets = df[amenities_col].map(parse_amenities_json)

    indicators = pd.DataFrame(
        {a: amenities_sets.map(lambda s, a=a: int(a in s)) for a in amenities_list},
        index=df.index,
    )
    # concat
    out = pd.concat([df, indicators], axis=1)

    return out


amenities_df = add_amenity_indicator_columns(amenities_df, amenitiesList)
print(amenities_df.shape)

(7274, 133)


In [15]:
amenities_df = amenities_df.drop(["amenities"], axis=1)

In [16]:
amenities_df.head()

,id,Gym,Heating,Hot water,Air conditioning,Iron,Dryer,Carbon monoxide alarm,Building staff,Luggage dropoff allowed,...,Home theater,Espresso machine,Balcony,Massage table,Courtyard,Day bed,Lounge area,Ski-in/Ski-out,Suitable for events,Smoking allowed
0,958,0,1,1,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,5858,0,1,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,7918,0,1,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,8142,0,1,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8339,0,1,1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


### Calendar Data Frame

In [17]:
sf_cal.head()

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,25662,2020-09-08,t,$125.00,$125.00,4.0,180.0
1,25662,2020-09-09,t,$125.00,$125.00,4.0,180.0
2,25662,2020-09-10,t,$126.00,$126.00,4.0,180.0
3,25662,2020-09-11,t,$139.00,$139.00,4.0,180.0
4,25662,2020-09-12,t,$151.00,$151.00,4.0,180.0


This Data Frame has folowing columns:
* __listing_id__ — id values, we'll use to join tables
* __date__ — we need to change datatype to datetime
* __available__ — it has to be boolean, so we need to change it
* __minimum_nights, maximum_nights__ — we have same columns in Listing Data Frame, drop they later
* __adjusted_price__, __price__ — target values

In [18]:
#converting datatype of price and adjusted_price columns to integer
sf_cal["price"] = sf_cal["price"].apply(lambda string: stringToNumConverter(string))
sf_cal["price"] = pd.to_numeric(sf_cal["price"], downcast="integer")

sf_cal["adjusted_price"] = sf_cal["adjusted_price"].apply(lambda string: stringToNumConverter(string))
sf_cal["adjusted_price"] = pd.to_numeric(sf_cal["adjusted_price"], downcast="integer")

#converting datatype of date columns to datetime
sf_cal["date"] = pd.to_datetime(arg=sf_cal["date"], errors="coerce")

#converting t, f value to Boolean datatype
sf_cal["available"] = sf_cal["available"].apply((lambda string: True if string == "t" else False))

In [19]:
print("Calendar Data Frame")
printColunmsInfo(sf_cal)

Calendar Data Frame
<class 'pandas.core.frame.DataFrame'>
Index: 2390652 entries, 0 to 2390651
Data columns (total 7 columns):
 #   Column          Dtype         
---  ------          -----         
 0   listing_id      int64         
 1   date            datetime64[ns]
 2   available       bool          
 3   price           float64       
 4   adjusted_price  float64       
 5   minimum_nights  float64       
 6   maximum_nights  float64       
dtypes: bool(1), datetime64[ns](1), float64(4), int64(1)
memory usage: 130.0 MB


Total columns:  7
Total rows:  2390652
---------------------------------------


## Data Understanding

Prior exploratory analysis showed that price is strongly associated with location and a small set of property-level amenities, including building staff, air conditioning, gym access, and private entrance. At the same time, widely available amenities such as kitchen, laptop-friendly workspace, and cooking basics showed weak or negative correlation with price, suggesting they function more as baseline expectations than price drivers.

Review scores demonstrated weak correlation with price while showing strong correlation with each other. This indicated that review metrics primarily reflect guest satisfaction rather than pricing power.

Together, these findings suggested that static correlations explain only part of pricing behavior. Many features appear important in isolation but lose influence once market conditions shift. For this reason, the current analysis treats these correlations as context rather than conclusions and focuses on how price signals behave over time.

Listings that accommodate more guests tend to command higher prices, with similar patterns observed for bedrooms and beds. Bathrooms show a much weaker relationship with price and contribute less to overall variation.

Location exerts a stronger influence on price than most property features. The highest prices cluster around areas near Golden Gate Park and the Financial District. Despite very different neighborhood characteristics, both areas sustain high prices, indicating that proximity and demand outweigh local conditions.

I then examined price changes over the twelve month period. During COVID, the average nightly price increased by about 33 dollars and followed a steady upward trend rather than short-term spikes.

Demand moved in the opposite direction. September of the prior year showed high occupancy, likely driven by seasonal travel. After that point, availability increased month by month. The sharpest decline occurred in spring, when nearly half of listings sat vacant by May.

These trends reveal a shift in market dynamics. Prices rose while demand fell. Hosts appeared to raise prices despite lower occupancy, possibly to offset reduced volume. By the end of the period, neither price nor demand had returned to pre-COVID levels.

This instability motivated the modeling stage. I prepared the data to test whether listing features could still explain price differences under changing conditions, rather than assuming patterns observed in stable periods would continue to hold.

## Prepare Data

#### Working with NaNs and categorical variables

Let's turn "last_review" and "host_since" from date type to categorical values. For that, we create new columns and fill them in.

In [20]:
sf_list["since_last_review"] = sf_list["last_review"].apply(lambda row : dateToCategorical(row))
sf_list["host_since_cat"] = sf_list["host_since"].apply(lambda row : dateToCategorical(row))

In [21]:
#drop all Nans in "price" columns
sf_list = sf_list.dropna(subset=["price"], axis=0)

In [22]:
def log_transform_numeric_columns(df: pd.DataFrame, columns: list[str], replacement_value: float = 0.0) -> pd.DataFrame:
    """
    Apply log1p transformation to selected numeric columns.
    Handles NaN and zero values explicitly.
    """
    result_df = df.copy()

    for col in columns:
        result_df[col] = result_df[col].fillna(0.0)
        result_df[col] = result_df[col].astype("float64")
        # Replace NaN with 0 for log1p
        result_df[col] = result_df[col].fillna(replacement_value)
        # Replace negative values if any
        result_df.loc[result_df[col] < 0, col] = replacement_value
        # Apply log1p
        result_df[col] = np.log1p(result_df[col])

    return result_df

num_cols = ["accommodates", "availability_30", "number_of_reviews", "host_listings_count", "reviews_per_month"]

# Apply the transformation
log_sf_list = log_transform_numeric_columns(sf_list, num_cols)

In [23]:
df = log_sf_list.copy()

# bathrooms, bedrooms, beds
num_room_cols = ["bathrooms", "bedrooms", "beds"]

for col in num_room_cols:
    df[col] = df.groupby("room_type")[col].transform(
        lambda x: x.fillna(x.median())
    )
    df[col] = df[col].fillna(df[col].median())

# review score columns
review_cols = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_checkin",
    "review_scores_value"
]

df["has_reviews"] = (df["number_of_reviews"] > 0).astype(int)

for col in review_cols:
    df[col] = df[col].fillna(0.0)

df[num_room_cols + review_cols + ["has_reviews"]].isna().sum()

bathrooms                      0
bedrooms                       0
beds                           0
review_scores_rating           0
review_scores_accuracy         0
review_scores_cleanliness      0
review_scores_communication    0
review_scores_location         0
review_scores_checkin          0
review_scores_value            0
has_reviews                    0
dtype: int64

In [24]:
cal_price_stats = (
    sf_cal
    .dropna(subset=["price"])
    .groupby("listing_id")["price"]
    .agg(
        days_with_price="count",
        unique_prices="nunique",
        min_price="min",
        median_price="median",
        max_price="max",
        std_price="std"
    )
    .reset_index()
)
print("Listings with calendar prices:", cal_price_stats.shape[0])
print("Share with >1 unique calendar price:",
      (cal_price_stats["unique_prices"] > 1).mean())

print("Example of listings with price changes:")
print(cal_price_stats.sort_values("unique_prices", ascending=False).head(10))

# 2) Сравнение sf_list.price и sf_cal.price
list_prices = df[["id", "price"]].rename(columns={"id": "listing_id", "price": "listing_price"})

merged = sf_cal.merge(list_prices, on="listing_id", how="left")

cmp = merged.dropna(subset=["price", "listing_price"]).copy()
cmp["diff"] = cmp["price"] - cmp["listing_price"]
cmp["abs_diff"] = cmp["diff"].abs()
cmp["same_price"] = (cmp["abs_diff"] < 1e-9)

print("Rows with both prices:", cmp.shape[0])
print("Share where calendar price equals listing price:", cmp["same_price"].mean())
print("Median absolute difference:", cmp["abs_diff"].median())
print("Mean absolute difference:", cmp["abs_diff"].mean())

cmp2 = sf_cal.dropna(subset=["price", "adjusted_price"]).copy()
cmp2["adj_diff"] = cmp2["adjusted_price"] - cmp2["price"]
print("Calendar adjusted_price vs price")
print("Share where adjusted_price != price:", (cmp2["adj_diff"].abs() > 1e-9).mean())
print("Median abs difference:", cmp2["adj_diff"].abs().median())

if "available" in sf_cal.columns:
    avail_check = sf_cal.assign(has_price=sf_cal["price"].notna()).groupby("available")["has_price"].mean()
    print("Share of non-null calendar price by available flag:")
    print(avail_check)

Listings with calendar prices: 11838
Share with >1 unique calendar price: 0.7497043419496536
Example of listings with price changes:
      listing_id  days_with_price  unique_prices  min_price  median_price  \
2093     7676566              298            245      549.0         910.5   
7454    32812271              278            241      630.0        1119.0   
308       439703              298            227      201.0         568.0   
6819    30531089              296            226      549.0        1106.5   
4221    19753187              299            222      295.0         500.0   
6877    30770207              297            219      316.0         547.0   
704      1212746              296            219      525.0         894.5   
2499     9990865              296            217      616.0         901.5   
7593    33271309              296            206      313.0         572.5   
2884    12614397              295            189      259.0         417.0   

      max_price   s

Exploratory analysis confirmed that calendar-level prices vary substantially over time. Approximately three quarters of listings exhibit multiple unique prices within a twelve month window, with large intra-listing price ranges. Listing-level prices match calendar prices in only about half of observations, indicating that the listing snapshot price represents a baseline rather than the effective daily price. As a result, calendar price was selected as the modeling target, and the data grain was defined at the listing and date level.

In [25]:
sf_cal["date"] = pd.to_datetime(sf_cal["date"])
cal_cols = ["listing_id", "date", "available", "price"]
sf_cal_model = sf_cal[cal_cols].copy()

# drop rows with missing target
sf_cal_model = sf_cal_model.dropna(subset=["price"])

list_df = df.copy()
list_df = list_df.drop(columns=["price"], axis=1)
list_df = list_df.rename(columns={"id": "listing_id"})

list_df = list_df.sort_values("listing_id").drop_duplicates(subset=["listing_id"], keep="first")

amen_df = amenities_df.copy()
amen_df = amen_df.rename(columns={"id": "listing_id"})
amen_df = amen_df.sort_values("listing_id").drop_duplicates(subset=["listing_id"], keep="first")

# merge
cal_plus_list = sf_cal_model.merge(list_df, on="listing_id", how="left", validate="m:1")
full_calendar_table = cal_plus_list.merge(amen_df, on="listing_id", how="left", validate="m:1")

print("Result shape:", full_calendar_table.shape)
print("Unique listings:", full_calendar_table["listing_id"].nunique())
print("Date range:", full_calendar_table["date"].min(), "to", full_calendar_table["date"].max())

missing_listing_rows = full_calendar_table["host_is_superhost"].isna().mean() if "host_is_superhost" in full_calendar_table.columns else None
print("Share of rows missing listing features (example):", missing_listing_rows)

Result shape: (2390596, 167)
Unique listings: 11838
Date range: 2019-09-12 00:00:00 to 2020-09-30 00:00:00
Share of rows missing listing features (example): 0.22855262871685555


I built a compact feature set before modeling to reduce noise and make results easier to interpret. I kept variables that describe location, listing structure, demand signals, host characteristics, and review quality because they have a direct and defensible link to price. I removed identifiers and raw timestamps because they do not represent underlying pricing drivers. I also removed high-missing or unstable operational fields, such as response time and response rates, since they can add volatility without improving generalization. For time-related signals, I prefer derived categorical features such as host_since_cat and since_last_review instead of raw dates, because derived features capture the same information in a form models handle more reliably. This selection creates a strong baseline feature set and supports fair comparisons when testing more complex models later.

In [26]:
df = full_calendar_table.copy()

y = df["price"].astype("float64")

df["month"] = df["date"].dt.month.astype("int16")
df["day_of_week"] = df["date"].dt.dayofweek.astype("int16")
df["is_weekend"] = (df["day_of_week"] >= 5).astype("int8")
df["day_of_year"] = df["date"].dt.dayofyear.astype("int16")

drop_cols = [
    "price",          # target
    "listing_id",     # avoid memorization by id
]

for c in ["host_since", "last_review"]:
    if c in df.columns:
        drop_cols.append(c)

X = df.drop(columns=[c for c in drop_cols if c in df.columns])

## Data Modeling

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pandas.api.types import CategoricalDtype
from sklearn.preprocessing import StandardScaler

### Baseline

In [28]:
y = df["price"].astype("float64")
X = df.drop(columns=["price"]).copy()

In [29]:
from pandas.api.types import is_bool_dtype, is_numeric_dtype

def is_binary_series(s: pd.Series) -> bool:
    s = s.dropna()
    if s.empty:
        return False
    # bool
    if is_bool_dtype(s):
        return s.nunique() > 1
    # "0"/"1"/"True"/"False"
    if s.dtype == "object":
        normalized = s.astype(str).str.strip().str.lower()
        uniques = set(normalized.unique())
        allowed = {"0", "1", "true", "false"}
        return uniques.issubset(allowed) and len(uniques) > 1
    # numeric: 0/1 (0.0/1.0)
    if is_numeric_dtype(s):
        vals = pd.to_numeric(s, errors="coerce").dropna().unique()
        uniques = set(np.unique(vals))
        return uniques.issubset({0, 1}) and len(uniques) > 1

    return False

In [30]:
X = X.loc[:, ~X.columns.duplicated()].copy()

def uniq(seq):
    return list(dict.fromkeys(seq))

# Baseline features
baseline_cols = [
    "month", "day_of_week", "is_weekend", "day_of_year",
    "room_type", "property_type", "neighbourhood_cleansed",
    "latitude", "longitude",
    "accommodates", "bathrooms", "bedrooms", "beds",
    "minimum_nights", "maximum_nights",
    "host_is_superhost", "host_listings_count",
    "host_response_time", "host_response_rate", "host_acceptance_rate",
    "since_last_review", "host_since_cat",
]
baseline_cols = [c for c in baseline_cols if c in X.columns]

from pandas.api.types import CategoricalDtype
categorical_cols = [
    c for c in baseline_cols
    if (X[c].dtype == "object") or isinstance(X[c].dtype, CategoricalDtype)
]

# Amenity columns
NON_AMENITY_BINARY = {
    "available", "has_reviews", "instant_bookable", "requires_license", "is_business_travel_ready",
}
non_amenity = set(baseline_cols + ["listing_id", "id", "date", "host_since", "last_review"] + list(NON_AMENITY_BINARY))

amenity_candidates = [c for c in X.columns if c not in non_amenity]

amenity_cols = []
for c in amenity_candidates:
    col = X[c]
    if isinstance(col, pd.DataFrame):
        col = col.iloc[:, 0]
    if is_binary_series(col):
        amenity_cols.append(c)

amenity_cols = uniq(amenity_cols)

# 1) keep_cols без повторного добавления categorical_cols
keep_cols = uniq(["date"] + baseline_cols + amenity_cols)
keep_cols = [c for c in keep_cols if c in X.columns]

print("keep_cols:", len(keep_cols))
print("Amenities:", len(amenity_cols))
print("Baseline:", len(baseline_cols), "Categorical:", len(categorical_cols))

X_baseline = X.loc[:, keep_cols].copy()

print("X_baseline shape:", X_baseline.shape)
print("Date range:", X_baseline["date"].min(), "to", X_baseline["date"].max())

keep_cols: 154
Amenities: 131
Baseline: 22 Categorical: 6
X_baseline shape: (2390596, 154)
Date range: 2019-09-12 00:00:00 to 2020-09-30 00:00:00


In [31]:
numeric_cols = [c for c in baseline_cols if c not in categorical_cols]

assert "date" in X_baseline.columns

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

amenity_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value=0))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
        ("amenities", amenity_transformer, amenity_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("num:", len(numeric_cols), "cat:", len(categorical_cols), "amenities:", len(amenity_cols))

num: 16 cat: 6 amenities: 131


In [32]:
print("overlap num-cat", set(numeric_cols) & set(categorical_cols))
print("overlap num-amen", set(numeric_cols) & set(amenity_cols))
print("overlap cat-amen", set(categorical_cols) & set(amenity_cols))

overlap num-cat set()
overlap num-amen set()
overlap cat-amen set()


In [33]:
X_baseline = X_baseline.copy()
X_baseline["date"] = pd.to_datetime(X_baseline["date"])

train_end = pd.Timestamp("2020-05-31")
val_end   = pd.Timestamp("2020-07-31")

train_mask = X_baseline["date"] <= train_end
val_mask   = (X_baseline["date"] > train_end) & (X_baseline["date"] <= val_end)
test_mask  = X_baseline["date"] > val_end

X_train = X_baseline.loc[train_mask].drop(columns=["date"])
X_val   = X_baseline.loc[val_mask].drop(columns=["date"])
X_test  = X_baseline.loc[test_mask].drop(columns=["date"])

y_train = y.loc[train_mask]
y_val   = y.loc[val_mask]
y_test  = y.loc[test_mask]

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
print("train dates:", X_baseline.loc[train_mask, "date"].min(), "to", X_baseline.loc[train_mask, "date"].max())
print("val dates:", X_baseline.loc[val_mask, "date"].min(), "to", X_baseline.loc[val_mask, "date"].max())
print("test dates:", X_baseline.loc[test_mask, "date"].min(), "to", X_baseline.loc[test_mask, "date"].max())

train: (1735123, 153) val: (367872, 153) test: (287601, 153)
train dates: 2019-09-12 00:00:00 to 2020-05-31 00:00:00
val dates: 2020-06-08 00:00:00 to 2020-07-31 00:00:00
test dates: 2020-08-15 00:00:00 to 2020-09-30 00:00:00


In [34]:
# Baseline model
model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]
)

y_train = y_train.squeeze()
y_test = y_test.squeeze()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

abs_err = np.abs(y_test.to_numpy() - y_pred)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nBaseline performance on future period")
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R2:", round(r2, 3))

print("\nError distribution (absolute)")
print("Median:", round(float(np.median(abs_err)), 3))
print("90th pct:", round(float(np.quantile(abs_err, 0.90)), 3))
print("99th pct:", round(float(np.quantile(abs_err, 0.99)), 3))


Baseline performance on future period
MAE: 164.025
RMSE: 664.549
R2: 0.182

Error distribution (absolute)
Median: 85.889
90th pct: 243.359
99th pct: 946.865


In [35]:
def evaluate_and_log(
    results_df: pd.DataFrame,
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
    extra: dict | None = None
) -> pd.DataFrame:
    y_true = y_true.squeeze()

    abs_err = np.abs(y_true.to_numpy() - y_pred)

    row = {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "AbsErr_Median": float(np.median(abs_err)),
        "AbsErr_P90": float(np.quantile(abs_err, 0.90)),
        "AbsErr_P99": float(np.quantile(abs_err, 0.99)),
        "n_test": int(len(y_true)),
    }

    if extra:
        row.update(extra)

    return pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

results = pd.DataFrame(columns=[
    "model", "MAE", "RMSE", "R2", "AbsErr_Median", "AbsErr_P90", "AbsErr_P99", "n_test",
    "alpha", "l1_ratio", "target_transform"
])

results = evaluate_and_log(
    results,
    "Ridge baseline",
    y_test,
    y_pred,
    extra={"alpha": 1.0, "l1_ratio": np.nan, "target_transform": "none"}
)

C:\Users\kitam\AppData\Local\Temp\ipykernel_6368\504678767.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)


print(baseline_model)
pre = baseline_model.named_steps.get("preprocess")
print(pre)

The baseline model demonstrates moderate predictive performance on a future time period, which is consistent with expectations for a linear approach using basic features and amenities.

The mean absolute error of 164 indicates the typical deviation between predictions and true values. This level of error is noticeable but acceptable for a baseline reference.

The root mean squared error of 665 is substantially larger than the MAE, suggesting the presence of occasional large prediction errors. This behavior points to a heavy-tailed error distribution, where most predictions are reasonable but a small fraction deviates strongly.

An R2 score of 0.182 shows that the model explains approximately 18 percent of the variance in the target variable on unseen future data. This level of explanatory power is limited but expected for a simple linear model without nonlinear effects or feature interactions.

The error distribution provides additional insight. The median absolute error of 86 indicates that half of the predictions fall within a relatively narrow error range. The 90th percentile at 243 marks the point where prediction quality begins to deteriorate more visibly. The 99th percentile close to 947 highlights rare but severe prediction failures.

Overall, the results indicate that the baseline captures broad trends and average effects but struggles with atypical listings and extreme cases. The current feature set provides useful signal, yet the linear structure constrains performance. This baseline therefore serves as a lower bound for model quality and a reference point for evaluating more expressive modeling approaches.

### ElasticNet + log1p(y)

In [36]:
from sklearn.linear_model import ElasticNet

In [37]:
y_train_s = y_train.squeeze()
y_val_s   = y_val.squeeze()
y_test_s  = y_test.squeeze()
# log-transform
y_train_log = np.log1p(y_train_s)
y_val_log   = np.log1p(y_val_s)
y_test_log  = np.log1p(y_test_s)

In [38]:
model_en = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000, random_state=42)),
    ]
)

model_en.fit(X_train, y_train_log)

y_pred_test_log = model_en.predict(X_test)
y_pred_test = np.expm1(y_pred_test_log)

abs_err = np.abs(y_test_s.to_numpy() - y_pred_test)

mae = mean_absolute_error(y_test_s, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test_s, y_pred_test))
r2 = r2_score(y_test_s, y_pred_test)

print("\nElasticNet + log1p(y) performance on future period")
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R2:", round(r2, 3))

print("\nError distribution (absolute)")
print("Median:", round(float(np.median(abs_err)), 3))
print("90th pct:", round(float(np.quantile(abs_err, 0.90)), 3))
print("99th pct:", round(float(np.quantile(abs_err, 0.99)), 3))


ElasticNet + log1p(y) performance on future period
MAE: 141.614
RMSE: 731.765
R2: 0.008

Error distribution (absolute)
Median: 53.654
90th pct: 175.263
99th pct: 1179.439


In [39]:
results = evaluate_and_log(
    results,
    "ElasticNet + log1p(y)",
    y_test_s,
    y_pred_test,
    extra={"alpha": 0.1, "l1_ratio": 0.5, "target_transform": "log1p"}
)

results_view = results.copy()
for col in ["MAE", "RMSE", "R2", "AbsErr_Median", "AbsErr_P90", "AbsErr_P99"]:
    results_view[col] = results_view[col].astype(float).round(3)

print("\nModel comparison table")
print(results_view.sort_values(by="RMSE"))


Model comparison table
                   model      MAE     RMSE     R2  AbsErr_Median  AbsErr_P90  \
0         Ridge baseline  164.025  664.549  0.182         85.889     243.359   
1  ElasticNet + log1p(y)  141.614  731.765  0.008         53.654     175.263   

   AbsErr_P99  n_test  alpha  l1_ratio target_transform  
0     946.865  287601    1.0       NaN             none  
1    1179.439  287601    0.1       0.5            log1p  


In [40]:
y_train_log = np.log1p(y_train_s.to_numpy())
y_val_true = y_val_s.to_numpy()

X_train_tr = preprocessor.fit_transform(X_train)
X_val_tr = preprocessor.transform(X_val)
X_test_tr  = preprocessor.transform(X_test)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

alphas = [1e-3, 1e-2, 1e-1]
l1_ratios = [0.2, 0.5, 0.8]

best_alpha, best_l1, best_rmse = None, None, np.inf

for a in alphas:
    for l1 in l1_ratios:
        en = ElasticNet(
            alpha=a,
            l1_ratio=l1,
            max_iter=10000,
            selection="random",
            random_state=42
        )
        en.fit(X_train_tr, y_train_log)

        pred_val = np.expm1(en.predict(X_val_tr))
        score = rmse(y_val_true, pred_val)

        if score < best_rmse:
            best_rmse = score
            best_alpha, best_l1 = a, l1

print("Best on val (RMSE):", round(best_rmse, 3), "alpha:", best_alpha, "l1_ratio:", best_l1)

Best on val (RMSE): 717.978 alpha: 0.001 l1_ratio: 0.2


## Evaluate the Results

Let's take a look at our results and compare them with each other.

metrics_df

The AdaBoost regressor showed bad r2 score. The predictions of this model are not similar to the real values.
Gradient Boosting and Extreme Gradient Boosting showed similar results, but Gradient Boosting is slightly better.
Finally, I trained a neural network that performs worse than Gradient Boosting and shows overfitting.

## Thank you!

In [41]:
en = ElasticNet(alpha=1e-3, l1_ratio=0.2, max_iter=3000, tol=1e-3, selection="random", random_state=42)
en.fit(X_train_tr, np.log1p(y_train_s.to_numpy()))
y_pred_test = np.expm1(en.predict(X_test_tr))

In [42]:
y_test_true = y_test_s.squeeze().to_numpy()

abs_err = np.abs(y_test_true - y_pred_test)

mae = mean_absolute_error(y_test_true, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test_true, y_pred_test))
r2 = r2_score(y_test_true, y_pred_test)

print("\nElasticNet fast + log1p(y) performance on future period")
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R2:", round(r2, 3))

print("\nError distribution (absolute)")
print("Median:", round(float(np.median(abs_err)), 3))
print("90th pct:", round(float(np.quantile(abs_err, 0.90)), 3))
print("99th pct:", round(float(np.quantile(abs_err, 0.99)), 3))

# results
row = {
    "model": "ElasticNet fast + log1p(y)",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "AbsErr_Median": float(np.median(abs_err)),
    "AbsErr_P90": float(np.quantile(abs_err, 0.90)),
    "AbsErr_P99": float(np.quantile(abs_err, 0.99)),
    "n_test": int(len(y_test_true)),
    "alpha": 1e-3,
    "l1_ratio": 0.2,
    "target_transform": "log1p",
}

results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)

results_view = results.copy()
for col in ["MAE", "RMSE", "R2", "AbsErr_Median", "AbsErr_P90", "AbsErr_P99"]:
    results_view[col] = results_view[col].astype(float).round(3)

print("\nModel comparison table")
print(results_view.sort_values(by="RMSE"))


ElasticNet fast + log1p(y) performance on future period
MAE: 124.215
RMSE: 715.348
R2: 0.052

Error distribution (absolute)
Median: 37.34
90th pct: 152.3
99th pct: 1034.168

Model comparison table
                        model      MAE     RMSE     R2  AbsErr_Median  \
0              Ridge baseline  164.025  664.549  0.182         85.889   
2  ElasticNet fast + log1p(y)  124.215  715.348  0.052         37.340   
1       ElasticNet + log1p(y)  141.614  731.765  0.008         53.654   

   AbsErr_P90  AbsErr_P99  n_test  alpha  l1_ratio target_transform  
0     243.359     946.865  287601  1.000       NaN             none  
2     152.300    1034.168  287601  0.001       0.2            log1p  
1     175.263    1179.439  287601  0.100       0.5            log1p  


The results highlight a clear trade-off between performance on typical observations and behavior in the tail of the target distribution.

The fast ElasticNet model with a log1p target transformation achieves the lowest MAE and median absolute error among all linear models. MAE decreases from 164 in the Ridge baseline to 124, while the median absolute error drops from 85.9 to 37.3. This represents a substantial improvement in predictive accuracy for the majority of listings.

The 90th percentile error also improves markedly, decreasing from 243 to 152, indicating more reliable predictions across most of the data.

However, RMSE increases relative to the Ridge baseline, and the R2 score remains low. This deterioration is driven by performance on extreme target values. The 99th percentile absolute error worsens compared to Ridge, suggesting systematic underestimation of high values after reversing the logarithmic transformation.

This pattern indicates that the log transformation and ElasticNet regularization effectively reduce noise and multicollinearity, leading to more stable and accurate predictions in the center of the distribution. At the same time, these mechanisms compress prediction ranges and limit the model’s ability to capture extreme outcomes. As a result, typical errors improve significantly, while tail performance degrades, which disproportionately affects RMSE and variance-based metrics such as R2.

Overall, the results suggest that linear models have reached their practical limit for this task. While they provide strong control over average prediction error, they are not well suited to modeling rare, high-value cases without explicit nonlinear structure. This motivates the transition to tree-based and boosting models, which are better equipped to capture nonlinear effects, feature interactions, and heavy-tailed target distributions.

### HistGradientBoostingRegressor

In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

In [44]:
numeric_cols_hgb = numeric_cols 
categorical_cols_hgb = categorical_cols 
amenity_cols_hgb = amenity_cols

# numeric
numeric_transformer_hgb = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

# categorical -> Ordinal
categorical_transformer_hgb = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )),
    ]
)

# amenities 0/1
amenity_transformer_hgb = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ]
)

preprocessor_hgb = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_hgb, numeric_cols_hgb),
        ("cat", categorical_transformer_hgb, categorical_cols_hgb),
        ("amenities", amenity_transformer_hgb, amenity_cols_hgb),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

In [45]:
hgb = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=63,
    min_samples_leaf=50,
    l2_regularization=0.0,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42,
)

model_hgb = Pipeline(
    steps=[
        ("preprocess", preprocessor_hgb),
        ("model", hgb),
    ]
)

model_hgb.fit(X_train, y_train_s)

y_pred_test = model_hgb.predict(X_test)

In [46]:
y_test_true = y_test_s.to_numpy()
abs_err = np.abs(y_test_true - y_pred_test)

mae = mean_absolute_error(y_test_true, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test_true, y_pred_test))
r2 = r2_score(y_test_true, y_pred_test)

print("\nHistGradientBoosting performance on future period")
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R2:", round(r2, 3))

print("\nError distribution (absolute)")
print("Median:", round(float(np.median(abs_err)), 3))
print("90th pct:", round(float(np.quantile(abs_err, 0.90)), 3))
print("99th pct:", round(float(np.quantile(abs_err, 0.99)), 3))

row = {
    "model": "HistGradientBoosting",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "AbsErr_Median": float(np.median(abs_err)),
    "AbsErr_P90": float(np.quantile(abs_err, 0.90)),
    "AbsErr_P99": float(np.quantile(abs_err, 0.99)),
    "n_test": int(len(y_test_true)),
    "alpha": np.nan,
    "l1_ratio": np.nan,
    "target_transform": "none",
}

results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)

results_view = results.copy()
for col in ["MAE", "RMSE", "R2", "AbsErr_Median", "AbsErr_P90", "AbsErr_P99"]:
    results_view[col] = results_view[col].astype(float).round(3)

print("\nModel comparison table")
print(results_view.sort_values(by="RMSE"))


HistGradientBoosting performance on future period
MAE: 97.605
RMSE: 684.817
R2: 0.131

Error distribution (absolute)
Median: 29.233
90th pct: 99.631
99th pct: 616.136

Model comparison table
                        model      MAE     RMSE     R2  AbsErr_Median  \
0              Ridge baseline  164.025  664.549  0.182         85.889   
3        HistGradientBoosting   97.605  684.817  0.131         29.233   
2  ElasticNet fast + log1p(y)  124.215  715.348  0.052         37.340   
1       ElasticNet + log1p(y)  141.614  731.765  0.008         53.654   

   AbsErr_P90  AbsErr_P99  n_test  alpha  l1_ratio target_transform  
0     243.359     946.865  287601  1.000       NaN             none  
3      99.631     616.136  287601    NaN       NaN             none  
2     152.300    1034.168  287601  0.001       0.2            log1p  
1     175.263    1179.439  287601  0.100       0.5            log1p  


HistGradientBoosting delivers the most balanced performance among all evaluated models and confirms the need for nonlinear methods in this task.

Across all metrics reflecting typical prediction errors, HistGradientBoosting substantially outperforms the linear baselines. The MAE decreases to 97.6, representing an improvement of nearly 40 percent over the Ridge baseline and a clear gain over ElasticNet. The median absolute error drops to 29.2, indicating that half of the predictions deviate from the true values by less than 30 units, which is a strong practical result.

The 90th percentile absolute error improves dramatically as well, falling to 99.6 compared to 243 for Ridge and 152 for ElasticNet. This demonstrates high reliability for the vast majority of observations.

The most significant improvement appears in tail behavior. The 99th percentile absolute error decreases to 616, which is substantially lower than both Ridge and ElasticNet. Unlike linear models, HistGradientBoosting is able to capture rare, high target values more effectively without systematic underestimation.

While the RMSE remains slightly higher than that of the Ridge baseline, it is considerably lower than for both ElasticNet variants. This outcome is expected, as Ridge benefits from linear smoothing that can reduce variance-based metrics at the cost of larger typical errors. The R2 score for HistGradientBoosting exceeds that of ElasticNet but remains below Ridge, reflecting a more balanced trade-off between central accuracy and robustness to extreme values.

Overall, the results indicate that HistGradientBoosting provides the best practical performance for this problem. It minimizes typical prediction error, substantially reduces the error distribution tail, and generalizes well to future data. Compared to linear models, it avoids the trade-off between average accuracy and extreme-case failures, making it the strongest candidate for further tuning and for use as the primary model.

## Evaluate the Results

In [47]:
results

,model,MAE,RMSE,R2,AbsErr_Median,AbsErr_P90,AbsErr_P99,n_test,alpha,l1_ratio,target_transform
0,Ridge baseline,164.024570,664.548851,0.181683,85.888611,243.359301,946.865232,287601,1.000,NaN,none
1,ElasticNet + log1p(y),141.614442,731.765067,0.007773,53.654161,175.263347,1179.439441,287601,0.100,0.5,log1p
2,ElasticNet fast + log1p(y),124.215337,715.348319,0.051794,37.340261,152.299900,1034.167910,287601,0.001,0.2,log1p
3,HistGradientBoosting,97.604959,684.817483,0.131005,29.233371,99.630525,616.135764,287601,NaN,NaN,none


The evaluation compares four models using a strict time-based split, with all metrics reported on a future test period. This setup ensures that performance reflects true generalization rather than temporal leakage.

The Ridge baseline establishes a reference point. While it achieves the highest R² (0.18), this comes at the cost of large typical errors. Both MAE (164) and median absolute error (86) are high, indicating that the model sacrifices accuracy for most observations in order to minimize variance-driven metrics. The heavy tail of the error distribution remains substantial, with a 99th percentile absolute error close to 947.

The ElasticNet with log1p transformation demonstrates the expected trade-off of regularized linear models. Both ElasticNet variants significantly reduce MAE and median error relative to Ridge, showing improved performance for typical observations. The tuned fast version reduces MAE to 124 and median error to 37, a strong improvement in central accuracy. However, these gains come at the expense of tail performance. RMSE increases, R² drops sharply, and the 99th percentile error worsens compared to Ridge. This behavior indicates systematic underestimation of extreme target values after inverse log transformation and aggressive coefficient shrinkage.

The HistGradientBoosting model provides the most balanced and practically meaningful performance. It achieves the lowest MAE (97.6) and the lowest median absolute error (29.2), indicating substantial improvements for the majority of predictions. The 90th percentile error drops to approximately 100, less than half of the Ridge baseline, demonstrating strong robustness across most cases. Crucially, tail behavior improves significantly: the 99th percentile absolute error decreases to 616, outperforming all linear models by a wide margin.

While the RMSE and R² of HistGradientBoosting do not surpass the Ridge baseline, this outcome is expected. Ridge optimizes variance-based metrics by smoothing predictions, whereas HistGradientBoosting prioritizes accurate conditional predictions across heterogeneous regimes. The substantial reduction in typical and tail errors suggests that the tree-based model captures nonlinear relationships and feature interactions that linear models cannot represent.

Overall, the results indicate that linear models have reached their performance limit for this task. Regularization and target transformation improve central accuracy but exacerbate failures on extreme values. In contrast, HistGradientBoosting delivers superior practical performance by simultaneously reducing typical error and controlling tail risk, making it the strongest candidate for further tuning and for deployment in a real-world setting.

## Conclusion

This study evaluated a progression of models under a strict time-based validation framework, moving from linear baselines to nonlinear tree-based methods. The results demonstrate a clear performance hierarchy and highlight the limitations of linear approaches for this prediction task.

The Ridge baseline provides a useful reference but exhibits high typical errors, indicating limited practical accuracy despite a relatively strong R². ElasticNet models, particularly with a log-transformed target, improve performance for the central part of the distribution by reducing MAE and median error. However, these gains come at the cost of significantly worse behavior on extreme values, leading to higher RMSE and unstable variance-based metrics. This trade-off reflects the inherent constraints of linear models with regularization when faced with heavy-tailed targets and complex feature interactions.

HistGradientBoosting emerges as the most effective model overall. It achieves the lowest MAE and median absolute error, substantially improves performance at the 90th percentile, and dramatically reduces extreme errors at the 99th percentile. These improvements indicate that the model successfully captures nonlinear relationships and interactions between baseline features and amenities, while remaining robust to rare but influential cases.

Although Ridge attains the highest R², this metric alone proves insufficient for model selection in this context. From a practical perspective, minimizing typical error and controlling tail risk are more important than maximizing explained variance. Under these criteria, HistGradientBoosting offers the best balance between accuracy, robustness, and generalization to future data.

In conclusion, the experiments show that linear models have reached their practical ceiling for this problem. Tree-based boosting methods provide a decisive improvement and represent the most suitable foundation for further development. Future work should focus on targeted hyperparameter tuning of the boosting model, feature importance analysis, and potential extensions such as monotonic constraints or custom loss functions aligned with business objectives.

## Thank you